# High-dimensional image embeddings: IMSTE and UMAP

Choose dataset, graph, optimizer, and rendering settings, then compare IMSTE with UMAP on the same standardized samples. Multilevel coarsening uses IMSTE graph ranks; by default, PCA of pooled coarse-group features initializes its smallest graph. Both embeddings are displayed with `render_datashader_embedding` to keep dense plots readable.

The full gallery, including class-mixture graph analysis, is archived at `notebooks/archive/00_high_dim_mst_gallery_full.ipynb`.

In [1]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "src" / "high_dim_mst_gallery.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks" / "src"))
    sys.path.insert(0, str(repo_root))

## 1. Select parameters and fit IMSTE

Set the number of samples and tune the distance metric, FAMST, IMSTE, and multilevel settings below. With `METRIC = "cosine"`, IMSTE uses exact Prim graph construction even when `MST_METHOD = "famst"`. With `PCA_INIT = True`, PCA of the pooled features represented by the coarsest groups starts the multilevel layout. Set it to `False` to use a seeded random start there.

In [ ]:
from high_dim_mst_gallery import run_high_dim_mst_gallery
from IPython.display import display

# Dataset size
MAX_SAMPLES = 60_000
RANDOM_STATE = 42

# IMSTE graph and optimizer
MST_METHOD = "famst"  # or "prim" for exact graph construction
METRIC = "euclidean"  # or "cosine"; cosine uses exact Prim for IMSTE
DEVICE = "mps"  # IMSTE graph construction and optimization device
N_MSTS = 4
N_EPOCHS = 500
PCA_INIT = True  # Use PCA on the coarsest graph in the multilevel initializer.
NEGATIVE_RATIO = 1
FAMST_MAX_NEIGHBORS = 15

# Multilevel initializer (enabled by default in this notebook)
MULTILEVEL_INIT = True
MULTILEVEL_REDUCTION_RATIO = 4
MULTILEVEL_BASE_SIZE = 1024
MULTILEVEL_EPOCHS = 10

# UMAP comparison
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_EPOCHS = 200
REPULSION_STRENGTH = 2

# Datashader display
RENDER_WIDTH = 800
RENDER_HEIGHT = 800
RENDER_MIN_ALPHA = 40
RENDER_SPREAD_PX = 1
RENDER_BACKGROUND = "white"  # Change to "black" to compare contrast.

# Load the image datasets and fit IMSTE. Sample image grids and the helper's
# scatter plots are disabled; the comparison below uses Datashader instead.
gallery = run_high_dim_mst_gallery(
    max_samples=MAX_SAMPLES,
    n_msts=N_MSTS,
    n_epochs=N_EPOCHS,
    pca_init=PCA_INIT,
    mst_method=MST_METHOD,
    metric=METRIC,
    device=DEVICE,
    negative_ratio=NEGATIVE_RATIO,
    mst_max_neighbors=FAMST_MAX_NEIGHBORS,
    random_state=RANDOM_STATE,
    plot_sample_images=False,
    plot_embeddings=False,
    multilevel_init=MULTILEVEL_INIT,
    multilevel_reduction_ratio=MULTILEVEL_REDUCTION_RATIO,
    multilevel_base_size=MULTILEVEL_BASE_SIZE,
    multilevel_epochs=MULTILEVEL_EPOCHS,
)
display(gallery["summary"])
if gallery["load_errors"]:
    display(gallery["load_errors"])


## 2. Fit UMAP to the same samples

The gallery helper standardizes features before IMSTE. This cell applies the same standardization to each raw dataset before fitting UMAP, so both methods receive matching input rows and feature scaling.


In [ ]:
from time import perf_counter

import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler

UMAP_RESULTS = {}
UMAP_SECONDS = {}
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    X_standardized = StandardScaler().fit_transform(raw_X)
    started = perf_counter()
    embedding = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=METRIC,
        compatibility_layout=False,
        nearest_neighbor_backend="rinnd",
        init="recursive",
        random_state=RANDOM_STATE,
        n_epochs=UMAP_EPOCHS,
        repulsion_strength=REPULSION_STRENGTH,
        n_jobs=1,
    ).fit_transform(X_standardized)
    UMAP_RESULTS[dataset_name] = embedding
    UMAP_SECONDS[dataset_name] = perf_counter() - started
    print(f"{dataset_name}: UMAP fit in {UMAP_SECONDS[dataset_name] / 60:.1f} min")

comparison = gallery["summary"][["dataset", "samples", "input_features", "seconds"]].rename(
    columns={"seconds": "IMSTE seconds"}
)
comparison["UMAP seconds"] = comparison["dataset"].map(UMAP_SECONDS)
display(comparison)


MNIST (28×28): UMAP fit in 0.3 min


## 3. Compare the embeddings with Datashader

Each panel rasterizes all points at the selected image size. Adjust the rendering settings above to change the mark spread, opacity, resolution, or background.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from imste.visualization import render_datashader_embedding

DATASHADER_IMAGES = {}
IMSTE_SECONDS = gallery["summary"].set_index("dataset")["seconds"].to_dict()

for dataset_name, (_, labels) in gallery["datasets"].items():
    embeddings = {
        "IMSTE · multilevel start" if MULTILEVEL_INIT else "IMSTE · PCA start":
            gallery["embeddings"][dataset_name],
        "UMAP": UMAP_RESULTS[dataset_name],
    }
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    for ax, (method, embedding) in zip(axes, embeddings.items()):
        image = render_datashader_embedding(
            embedding,
            labels,
            width=RENDER_WIDTH,
            height=RENDER_HEIGHT,
            min_alpha=RENDER_MIN_ALPHA,
            background=RENDER_BACKGROUND,
            spread_px=RENDER_SPREAD_PX,
        )
        DATASHADER_IMAGES[(dataset_name, method)] = image
        ax.imshow(image.to_pil())
        elapsed = (
            IMSTE_SECONDS[dataset_name]
            if method.startswith("IMSTE")
            else UMAP_SECONDS[dataset_name]
        )
        runtime = f"{elapsed / 60:.1f} min" if elapsed >= 60 else f"{elapsed:.1f} s"
        ax.set_title(f"{dataset_name} — {method} ({runtime})")
        ax.set_axis_off()
    display(fig)
    plt.close(fig)


## 4. Save embeddings for interactive rendering

This writes one compact cache containing the labels, UMAP coordinates, and IMSTE coordinates for each successfully processed image dataset. Rerun this cell after changing the sample size or fit settings to replace the cached gallery.

In [ ]:
from pathlib import Path
import numpy as np

EMBEDDING_CACHE_DIR = repo_root / "data" / "embeddings"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
MNIST_EMBEDDINGS_PATH = EMBEDDING_CACHE_DIR / "mnist_gallery_embeddings.npz"
cache = {
    "dataset_names": np.asarray(list(gallery["datasets"]), dtype=str),
    "max_samples": np.asarray(MAX_SAMPLES),
    "random_state": np.asarray(RANDOM_STATE),
    "device": np.asarray(DEVICE),
    "multilevel_init": np.asarray(MULTILEVEL_INIT),
    "pca_init": np.asarray(PCA_INIT),
}
for index, (dataset_name, (_, labels)) in enumerate(gallery["datasets"].items()):
    cache[f"labels_{index}"] = np.asarray(labels)
    cache[f"imste_{index}"] = np.asarray(gallery["embeddings"][dataset_name], dtype=np.float32)
    cache[f"umap_{index}"] = np.asarray(UMAP_RESULTS[dataset_name], dtype=np.float32)
    cache[f"imste_seconds_{index}"] = np.asarray(
        float(gallery["summary"].set_index("dataset").loc[dataset_name, "seconds"])
    )
    cache[f"umap_seconds_{index}"] = np.asarray(float(UMAP_SECONDS[dataset_name]))
np.savez_compressed(MNIST_EMBEDDINGS_PATH, **cache)
print(f"Saved {len(gallery['datasets'])} datasets to {MNIST_EMBEDDINGS_PATH}")
